# Census data preparation

## Purpose
Build the TTWA-level socio-economic and urban-rural contextual variables
from 2011 Census and related LSOA data. These variables are used to colour
and interpret the Ball Mapper graph.

## Input
- Census 2011 LSOA tables from NOMIS: tenure, NS-SeC, highest qualification,
  economic activity, usual residents and area (hectares)
- 2011 Rural–Urban Classification (LSOA)
- LSOA-level house prices
- ONS 2011 LSOA–TTWA lookup

## Main steps
- Merge LSOA tables and map LSOAs to TTWAs (34,753 LSOAs across 173 England & Wales TTWAs)
- Aggregate tenure, NS-SeC and qualification percentages to TTWA level
  using household-weighted means
- Keep a reduced set of variables: % owned, % private rented,
  % higher managerial, % routine, % no qualifications, % Level 4+
- Calculate % self-employed (of economically active) and
  % economically inactive (of residents aged 16–74)
- Calculate population density (residents per hectare) and a
  log-transformed version to reduce right skew
- Build a population-weighted urban–rural score (1 = major conurbation,
  5 = rural village) and % of population living in urban LSOAs
- Calculate TTWA median house price from LSOA-level December 2011 prices,
  with coverage checked for each TTWA

## Output
`data/processed/census_filtered.csv`: 173 TTWAs with the contextual
variables used for Ball Mapper colouring.

In [1]:
import pandas as pd

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Starting with tenure, job type and quals Census 2011 datasets

In [2]:
tenure = pd.read_csv('/Users/scandimimi/Documents/Manchester/ERP/NOMIS_2011/Tenure.csv')

In [3]:
NEC = pd.read_csv('/Users/scandimimi/Documents/Manchester/ERP/NOMIS_2011/NEC.csv')

In [4]:
quals = pd.read_csv('/Users/scandimimi/Documents/Manchester/ERP/NOMIS_2011/Quals.csv')

In [5]:
merged = tenure.merge(NEC, on='mnemonic').merge(quals, on='mnemonic')

In [6]:
merged = tenure.merge(NEC, on=['mnemonic', '2011 super output area - lower layer']).merge(quals, on=['mnemonic', '2011 super output area - lower layer'])


In [7]:
qual_cols = [
    'No qualifications',
    'Level 1 qualifications',
    'Level 2 qualifications',
    'Apprenticeship',
    'Level 3 qualifications',
    'Level 4 qualifications and above',
    'Other qualifications'
]

total_col = 'All categories: Highest level of qualification'

for col in qual_cols:
    quals[col + ' %'] = (quals[col] / quals[total_col] * 100).round(1)


Importing the LSOA-TTWA 2011 ONS lookup table

In [8]:
lookup = pd.read_csv('/Users/scandimimi/Documents/Manchester/ERP/census/lookup.csv')

In [9]:
print(lookup.columns.tolist())
print(lookup.head(2))

['LSOA11CD', 'LSOA11NM', 'TTWA11CD', 'TTWA11NM', 'ObjectId']
   LSOA11CD   LSOA11NM   TTWA11CD   TTWA11NM  ObjectId
0  95DD04W1     Ballee  N12000001  Ballymena         1
1  95DD05W1  Ballykeel  N12000001  Ballymena         2


In [10]:
merged = merged.merge(lookup, left_on='mnemonic', right_on='LSOA11CD', how='left')


In [11]:
merged.head(2)

,2011 super output area - lower layer,mnemonic,All households,%_x,Owned,%.1_x,Owned: Owned outright,%.2_x,Owned: Owned with a mortgage or loan,%.3_x,...,Level 2 qualifications,Apprenticeship,Level 3 qualifications,Level 4 qualifications and above,Other qualifications,LSOA11CD,LSOA11NM,TTWA11CD,TTWA11NM,ObjectId
0,City of London 001A,E01000001,876.0,100.0,533.0,60.8,355.0,40.5,178.0,20.3,...,75.0,7.0,94.0,1047.0,67.0,E01000001,City of London 001A,E30000234,London,943.0
1,City of London 001B,E01000002,830.0,100.0,527.0,63.5,314.0,37.8,213.0,25.7,...,71.0,5.0,91.0,1024.0,31.0,E01000002,City of London 001B,E30000234,London,944.0


TTWA aggregation

In [12]:
merged = merged[merged['LSOA11CD'].notna()].copy()
print(f"Loaded {merged.shape[0]} LSOAs across {merged['TTWA11NM'].nunique()} TTWAs")

Loaded 34753 LSOAs across 173 TTWAs


In [13]:
# Tenure (suffix _x)
tenure_map = {
    '%.1_x': 'pct_owned',
    '%.2_x': 'pct_owned_outright',
    '%.3_x': 'pct_owned_mortgage',
    '%.5_x': 'pct_social_rented',
    '%.8_x': 'pct_private_rented',
    '%.11_x': 'pct_rent_free',
}
 
# NS-SeC (suffix _y)
nssec_map = {
    '%.1_y': 'pct_higher_managerial',
    '%.4_y': 'pct_lower_managerial',
    '%.5_y': 'pct_intermediate',
    '%.6_y': 'pct_small_employers',
    '%.7_y': 'pct_lower_supervisory',
    '%.8_y': 'pct_semi_routine',
    '%.9_y': 'pct_routine',
    '%.10_y': 'pct_never_worked',
}
 
for old, new in {**tenure_map, **nssec_map}.items():
    merged[new] = merged[old]

In [14]:
qual_count_cols = [
    'No qualifications',
    'Level 1 qualifications',
    'Level 2 qualifications',
    'Apprenticeship',
    'Level 3 qualifications',
    'Level 4 qualifications and above',
    'Other qualifications',
]
 
qual_total = merged[qual_count_cols].sum(axis=1)
 
merged['pct_no_quals']     = (merged['No qualifications']                    / qual_total * 100).round(1)
merged['pct_level4_plus']  = (merged['Level 4 qualifications and above']     / qual_total * 100).round(1)
merged['pct_level3']       = (merged['Level 3 qualifications']               / qual_total * 100).round(1)

In [15]:
pct_vars = (
    list(tenure_map.values()) +
    list(nssec_map.values()) +
    ['pct_no_quals', 'pct_level4_plus', 'pct_level3']
)
 
df_clean = merged[['LSOA11CD', 'LSOA11NM', 'TTWA11CD', 'TTWA11NM', 'All households'] + pct_vars]

In [16]:
def weighted_mean(group, value_col, weight_col):
    return (group[value_col] * group[weight_col]).sum() / group[weight_col].sum()
 
ttwa_rows = []
for ttwa_code, group in df_clean.groupby('TTWA11CD'):
    row = {
        'TTWA11CD': ttwa_code,
        'TTWA11NM': group['TTWA11NM'].iloc[0],
    }
    for var in pct_vars:
        row[var] = round(weighted_mean(group, var, 'All households'), 2)
    ttwa_rows.append(row)
 
ttwa_df = pd.DataFrame(ttwa_rows)

In [17]:
print(f"\nOutput shape: {ttwa_df.shape}")
print(f"Any nulls: {ttwa_df.isnull().sum().sum()}")
print("\nDescriptive statistics:")
print(ttwa_df[pct_vars].describe().round(2))


Output shape: (173, 19)
Any nulls: 0

Descriptive statistics:
       pct_owned  pct_owned_outright  pct_owned_mortgage  pct_social_rented  \
count     173.00              173.00              173.00             173.00   
mean       67.40               35.56               31.84              14.85   
std         3.69                5.55                4.26               3.66   
min        49.42               21.75               20.54               8.46   
25%        65.28               31.45               29.37              12.47   
50%        67.71               35.22               32.59              14.31   
75%        69.92               39.07               34.68              16.17   
max        76.46               53.94               39.63              25.47   

       pct_private_rented  pct_rent_free  pct_higher_managerial  \
count              173.00         173.00                 173.00   
mean                15.56           1.55                   9.17   
std                  2.7

In [20]:
census11 = ttwa_df

In [21]:
census11.head()

,TTWA11CD,TTWA11NM,pct_owned,pct_owned_outright,pct_owned_mortgage,pct_social_rented,pct_private_rented,pct_rent_free,pct_higher_managerial,pct_lower_managerial,pct_intermediate,pct_small_employers,pct_lower_supervisory,pct_semi_routine,pct_routine,pct_never_worked,pct_no_quals,pct_level4_plus,pct_level3
0,E30000004,Barnsley,64.24,30.29,33.96,21.16,12.57,1.59,6.12,17.05,12.18,8.69,8.58,17.32,17.98,6.33,32.37,17.29,11.38
1,E30000018,Bradford,64.45,28.47,35.98,15.24,18.24,1.53,7.17,17.14,12.80,8.84,7.06,14.62,13.51,10.29,28.19,20.47,11.29
2,E30000029,Halifax,66.55,31.22,35.33,15.23,16.41,1.36,9.77,20.99,13.01,9.46,7.05,14.88,12.55,5.81,23.83,24.99,12.26
3,E30000039,Skipton,73.12,42.30,30.83,9.01,15.39,2.02,11.19,24.44,11.60,15.17,7.07,13.15,9.44,2.27,20.41,31.58,12.25
4,E30000046,Dorchester and Weymouth,67.21,37.57,29.64,13.82,16.60,1.62,9.34,23.04,13.02,11.33,8.27,15.29,10.95,3.54,20.99,26.32,12.71


In [22]:
census_filtered = census11[
    [
        "TTWA11CD",
        "TTWA11NM",
        "pct_owned",
        "pct_private_rented",
        "pct_higher_managerial",
        "pct_routine",
        "pct_no_quals",
        "pct_level4_plus"
    ]
].copy()

Merging economic activity with lookup table

In [23]:
econ_act = pd.read_csv('/Users/scandimimi/Documents/Manchester/ERP/ColourationV/EconomicAct.csv')


In [27]:
econ_act = econ_act[
    econ_act["mnemonic"]
    .astype(str)
    .str.match(r"^[EW]\d{8}$", na=False)
].copy()

In [ ]:
econ_ttwa = econ_ttwa.drop(columns="LSOA11CD")

In [28]:
econ_ttwa = econ_act.merge(
    lookup[["LSOA11CD", "TTWA11CD", "TTWA11NM"]],
    left_on="mnemonic",
    right_on="LSOA11CD",
    how="left",
    validate="many_to_one"
)

In [29]:
count_cols = [
    "All usual residents aged 16 to 74",
    "Economically active",
    "Economically active: Self-employed",
    "Economically Inactive",
    "Unemployed: Age 16 to 24",
    "Unemployed: Age 50 to 74"
]

Aggregation to TTWA:

In [30]:
econ_ttwa = (
    econ_ttwa
    .groupby(["TTWA11CD", "TTWA11NM"], as_index=False)[count_cols]
    .sum()
)

Calculating self-employed as a % of the economically active population

In [31]:
econ_ttwa["pct_self_employed"] = (
    econ_ttwa["Economically active: Self-employed"] /
    econ_ttwa["Economically active"]
) * 100

In [32]:
econ_ttwa["TTWA11CD"].isna().sum()

np.int64(0)

In [33]:
econ_ttwa["pct_econ_inactive"] = (
    econ_ttwa["Economically Inactive"] /
    econ_ttwa["All usual residents aged 16 to 74"]
) * 100

In [34]:
census_filtered = census_filtered.merge(
    econ_ttwa[
        [
            "TTWA11CD",
            "pct_self_employed",
            "pct_econ_inactive"
        ]
    ],
    on="TTWA11CD",
    how="left",
    validate="one_to_one"
)

In [35]:
census_filtered.head(10)

,TTWA11CD,TTWA11NM,pct_owned,pct_private_rented,pct_higher_managerial,pct_routine,pct_no_quals,pct_level4_plus,pct_self_employed,pct_econ_inactive
0,E30000004,Barnsley,64.24,12.57,6.12,17.98,32.37,17.29,11.518735,33.403314
1,E30000018,Bradford,64.45,18.24,7.17,13.51,28.19,20.47,12.518259,33.274171
2,E30000029,Halifax,66.55,16.41,9.77,12.55,23.83,24.99,13.414623,29.491282
3,E30000039,Skipton,73.12,15.39,11.19,9.44,20.41,31.58,20.960896,28.240463
4,E30000046,Dorchester and Weymouth,67.21,16.60,9.34,10.95,20.99,26.32,15.258502,31.192578
5,E30000051,Falmouth,67.63,18.10,8.22,10.23,18.94,27.78,19.097547,34.665216
6,E30000054,Grantham,66.51,16.83,9.94,12.12,22.45,24.58,14.709801,28.551930
7,E30000061,Hastings,63.88,21.85,7.70,9.71,25.51,22.81,19.301294,34.506275
8,E30000064,Hexham,66.71,16.36,12.15,9.51,20.65,33.44,22.456805,30.149408
9,E30000070,Isle of Wight,70.09,17.41,7.74,11.41,24.30,22.72,17.990283,35.619116


Population Density merge

In [36]:
popdens = pd.read_csv('/Users/scandimimi/Documents/Manchester/ERP/ColourationV/PopDens.csv')

In [37]:
popdens.head(10)

,Rural Urban:,Total,Unnamed: 2,Unnamed: 3
0,2011 super output area - lower layer,mnemonic,All usual residents,Density (number of persons per hectare)
1,City of London 001A,E01000001,1465,112.9
2,City of London 001B,E01000002,1436,62.9
3,City of London 001C,E01000003,1346,227.7
4,City of London 001E,E01000005,985,52
5,Barking and Dagenham 016A,E01000006,1703,116.2
6,Barking and Dagenham 015A,E01000007,1391,69.5
7,Barking and Dagenham 015B,E01000008,1544,79.1
8,Barking and Dagenham 016B,E01000009,1773,138.6
9,Barking and Dagenham 015C,E01000010,2840,80.7


In [38]:
hectares = pd.read_csv('/Users/scandimimi/Documents/Manchester/ERP/ColourationV/Hectares.csv')

In [39]:
hectares = hectares[
    hectares["mnemonic"]
    .astype(str)
    .str.match(r"^[EW]\d{8}$", na=False)
].copy()

In [40]:
hectares_ttwa = hectares.merge(
    lookup[["LSOA11CD", "TTWA11CD", "TTWA11NM"]],
    left_on="mnemonic",
    right_on="LSOA11CD",
    how="left",
    validate="many_to_one"
)

In [41]:
hectares_ttwa["TTWA11CD"].isna().sum()

np.int64(0)

In [42]:
hectares_ttwa = (
    hectares_ttwa
    .groupby(["TTWA11CD", "TTWA11NM"], as_index=False)
    .agg(
        population=("All usual residents", "sum"),
        area_hectares=("Area Hectares", "sum")
    )
)

In [43]:
hectares_ttwa["pop_density"] = (
    hectares_ttwa["population"] /
    hectares_ttwa["area_hectares"]
)

In [44]:
hectares_ttwa["pop_density"].describe()

count    173.000000
mean       4.165446
std        4.828979
min        0.185492
25%        1.173509
50%        2.566172
75%        4.752388
max       36.870833
Name: pop_density, dtype: float64

In [45]:
census_filtered = census_filtered.merge(
    hectares_ttwa[["TTWA11CD", "pop_density"]],
    on="TTWA11CD",
    how="left",
    validate="one_to_one"
)

Making a log-transformed pop-dens variable due to right-skew

In [46]:
import numpy as np

census_filtered["log_pop_density"] = np.log1p(
    census_filtered["pop_density"]
)

In [47]:
census_filtered[
    ["TTWA11CD", "TTWA11NM", "pop_density", "log_pop_density"]
].head()

,TTWA11CD,TTWA11NM,pop_density,log_pop_density
0,E30000004,Barnsley,7.296448,2.115827
1,E30000018,Bradford,15.376720,2.795861
2,E30000029,Halifax,5.600814,1.887193
3,E30000039,Skipton,0.470609,0.385677
4,E30000046,Dorchester and Weymouth,1.687377,0.988566


Rural-Urban measure

In [48]:
rural = pd.read_csv("/Users/scandimimi/Documents/Manchester/ERP/ColourationV/RuralUrban.csv")

In [49]:
sorted(rural["RUC11CD"].dropna().unique())

['A1', 'B1', 'C1', 'C2', 'D1', 'D2', 'E1', 'E2']

Making sense of the ranking system

In [50]:
ruc_score = {
    "A1": 1,  # Urban major conurbation
    "B1": 2,  # Urban minor conurbation
    "C1": 3,  # Urban city and town
    "C2": 3,  # Urban city and town in a sparse setting
    "D1": 4,  # Rural town and fringe
    "D2": 4,  # Rural town and fringe in a sparse setting
    "E1": 5,  # Rural village and dispersed
    "E2": 5   # Rural village and dispersed in a sparse setting
}

In [51]:
rural["urban_rural_score"] = rural["RUC11CD"].map(ruc_score)

In [52]:
rural[["RUC11CD", "RUC11", "urban_rural_score"]].drop_duplicates().sort_values("urban_rural_score")

,RUC11CD,RUC11,urban_rural_score
0,A1,Urban major conurbation,1
6853,B1,Urban minor conurbation,2
451,C1,Urban city and town,3
13964,C2,Urban city and town in a sparse setting,3
2159,D1,Rural town and fringe,4
13955,D2,Rural town and fringe in a sparse setting,4
783,E1,Rural village and dispersed,5
13522,E2,Rural village and dispersed in a sparse setting,5


In [53]:
rural = rural.merge(
    hectares[["mnemonic", "All usual residents"]],
    left_on="LSOA11CD",
    right_on="mnemonic",
    how="left",
    validate="one_to_one"
)

In [54]:
rural_ttwa = rural.merge(
    lookup[["LSOA11CD", "TTWA11CD", "TTWA11NM"]],
    on="LSOA11CD",
    how="left",
    validate="many_to_one"
)

In [56]:
rural_ttwa["weighted_urban_rural"] = (
    rural_ttwa["urban_rural_score"] *
    rural_ttwa["All usual residents"]
)

In [57]:
rural_ttwa = (
    rural_ttwa
    .groupby(["TTWA11CD", "TTWA11NM"], as_index=False)
    .agg(
        weighted_score=("weighted_urban_rural", "sum"),
        population=("All usual residents", "sum")
    )
)

In [59]:
rural_ttwa["urban_rural_score"] = (
    rural_ttwa["weighted_score"] /
    rural_ttwa["population"]
)

In [60]:
rural_ttwa[
    ["TTWA11NM", "urban_rural_score"]
].sort_values("urban_rural_score")

,TTWA11NM,urban_rural_score
91,London,1.070029
62,Dudley,1.132555
28,Birkenhead,1.147763
96,Manchester,1.172803
86,Leeds,1.355580
...,...,...
83,Kingsbridge and Dartmouth,4.533023
41,Bude,4.605703
84,Launceston,4.650737
136,Wadebridge,4.659817


In [61]:
rural_ttwa["urban_rural_score"].describe()

count    173.000000
mean       3.386787
std        0.762562
min        1.070029
25%        3.204412
50%        3.468900
75%        3.794518
max        4.776730
Name: urban_rural_score, dtype: float64

In [62]:
census_filtered = census_filtered.merge(
    rural_ttwa[["TTWA11CD", "urban_rural_score"]],
    on="TTWA11CD",
    how="left",
    validate="one_to_one"
)

Getting a % Urban (Urbanity score)

In [63]:
rural["is_urban"] = rural["RUC11CD"].isin(
    ["A1", "B1", "C1", "C2"]
).astype(int)

rural["urban_population"] = (
    rural["is_urban"] *
    rural["All usual residents"]
)

In [64]:
urban_ttwa = rural.merge(
    lookup[["LSOA11CD", "TTWA11CD", "TTWA11NM"]],
    on="LSOA11CD",
    how="left",
    validate="many_to_one"
)

In [65]:
urban_ttwa = (
    urban_ttwa
    .groupby(["TTWA11CD", "TTWA11NM"], as_index=False)
    .agg(
        urban_population=("urban_population", "sum"),
        population=("All usual residents", "sum")
    )
)

In [66]:
urban_ttwa["pct_urban"] = (
    urban_ttwa["urban_population"] /
    urban_ttwa["population"]
) * 100

In [67]:
census_filtered = census_filtered.merge(
    urban_ttwa[["TTWA11CD", "pct_urban"]],
    on="TTWA11CD",
    how="left",
    validate="one_to_one"
)

House Price measure

In [68]:
houses = pd.read_csv("/Users/scandimimi/Documents/Manchester/ERP/ColourationV/HouseP2.csv")


In [69]:
houses.head

<bound method NDFrame.head of        LSOA code    March      Dec
0      E01011949   81,500   68,420
1      E01011950   56,750   45,000
2      E01011951   55,000   59,500
3      E01011952  106,945        :
4      E01011953   50,000   57,750
...          ...      ...      ...
34748  W01001320   94,000  110,000
34749  W01001321   75,400   75,375
34750  W01001322   74,250   70,000
34751  W01001324   86,000   98,000
34752  W01001898  140,000  147,000

[34753 rows x 3 columns]>

In [70]:
for col in ["March", "Dec"]:
    houses[col] = (
        houses[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

    houses[col] = pd.to_numeric(
        houses[col],
        errors="coerce"
    )

In [71]:
houses[["March", "Dec"]].isna().sum()

March    1188
Dec      1134
dtype: int64

In [72]:
houses_ttwa = houses.merge(
    lookup[["LSOA11CD", "TTWA11CD", "TTWA11NM"]],
    left_on="LSOA code",
    right_on="LSOA11CD",
    how="left",
    validate="many_to_one"
)

In [73]:
house_coverage = (
    houses_ttwa
    .groupby(["TTWA11CD", "TTWA11NM"])
    .agg(
        total_lsoas=("LSOA code", "size"),
        lsoas_with_price=("Dec", "count")
    )
    .reset_index()
)

house_coverage["pct_coverage"] = (
    house_coverage["lsoas_with_price"] /
    house_coverage["total_lsoas"]
) * 100

In [74]:
house_coverage.sort_values("pct_coverage").head(15)

,TTWA11CD,TTWA11NM,total_lsoas,lsoas_with_price,pct_coverage
90,E30000233,Liverpool,648,579,89.351852
153,K01000013,Newport,203,185,91.133005
159,W22000016,Pembroke and Tenby,24,22,91.666667
137,E30000283,Wakefield and Castleford,213,196,92.018779
8,E30000064,Hexham,26,24,92.307692
129,E30000275,Sunderland,263,243,92.395437
29,E30000169,Birmingham,1039,960,92.396535
142,E30000288,Wolverhampton and Walsall,466,434,93.133047
167,W22000029,Merthyr Tydfil,148,138,93.243243
73,E30000215,Hartlepool,62,58,93.548387


In [75]:
houses_ttwa_summary = (
    houses_ttwa
    .groupby(["TTWA11CD", "TTWA11NM"], as_index=False)
    .agg(
        median_house_price=("Dec", "median")
    )
)

In [76]:
houses_ttwa_summary = houses_ttwa_summary.merge(
    house_coverage[
        ["TTWA11CD", "pct_coverage"]
    ],
    on="TTWA11CD",
    how="left",
    validate="one_to_one"
)

In [77]:
houses_ttwa_summary.head()

,TTWA11CD,TTWA11NM,median_house_price,pct_coverage
0,E30000004,Barnsley,102500.0,97.468354
1,E30000018,Bradford,105000.0,95.207668
2,E30000029,Halifax,113500.0,96.875000
3,E30000039,Skipton,182250.0,100.000000
4,E30000046,Dorchester and Weymouth,198000.0,100.000000


In [78]:
houses_ttwa_summary["median_house_price"].describe()

count       173.000000
mean     163769.268786
std       45931.522894
min       78450.000000
25%      126000.000000
50%      160000.000000
75%      192500.000000
max      300000.000000
Name: median_house_price, dtype: float64

In [79]:
census_filtered = census_filtered.merge(
    houses_ttwa_summary[
        ["TTWA11CD", "median_house_price"]
    ],
    on="TTWA11CD",
    how="left",
    validate="one_to_one"
)

In [80]:
census_filtered["median_house_price"].isna().sum()

np.int64(0)

In [81]:
census_filtered.to_csv("census_filtered.csv", index=False)